In [1]:
import rasterio
import numpy as np
import cv2
from scipy.ndimage import uniform_filter

In [2]:
# =========================
# STEP 1: LOAD VV & VH
# =========================
def load_sar(vv_path, vh_path):
    with rasterio.open(vv_path) as src:
        vv = src.read(1).astype(np.float32)
        profile = src.profile

    with rasterio.open(vh_path) as src:
        vh = src.read(1).astype(np.float32)

    return vv, vh, profile

In [3]:
# =========================
# STEP 2: CONVERT TO dB
# =========================
def to_db(img):
    img = np.clip(img, 1e-6, None)
    return 10 * np.log10(img)

In [4]:
# =========================
# STEP 3: LEE FILTER
# =========================
def lee_filter(img, size=5):
    mean = uniform_filter(img, size)
    mean_sq = uniform_filter(img**2, size)
    variance = mean_sq - mean**2

    overall_var = np.var(img)

    weights = variance / (variance + overall_var + 1e-8)
    filtered = mean + weights * (img - mean)

    return filtered

In [5]:
# =========================
# STEP 4: NORMALIZATION
# =========================
def normalize(img):
    return (img - np.mean(img)) / (np.std(img) + 1e-8)

In [6]:
# =========================
# STEP 5: PROCESS PIPELINE
# =========================
def process_sar(vv, vh):
    # Convert to dB
    vv_db = to_db(vv)
    vh_db = to_db(vh)

    # Speckle filtering
    vv_f = lee_filter(vv_db)
    vh_f = lee_filter(vh_db)

    # Normalize
    vv_n = normalize(vv_f)
    vh_n = normalize(vh_f)

    # OPTIONAL: ratio channel (improves performance)
    ratio = vv_db - vh_db

    # Stack channels (choose 2 or 3)
    image = np.stack([vv_n, vh_n], axis=-1)
    # image = np.stack([vv_n, vh_n, ratio], axis=-1)  # if using 3 channels

    return image

In [7]:
# =========================
# STEP 6: RESIZE
# =========================
# def resize_image(image, size=(512, 512)):
#     resized = np.zeros((size[0], size[1], image.shape[2]), dtype=np.float32)

#     for i in range(image.shape[2]):
#         resized[:, :, i] = cv2.resize(image[:, :, i], size)

#     return resized

In [8]:
# =========================
# STEP 7: SAVE MULTI-BAND TIFF
# =========================
def save_tiff(output_path, image, profile):
    profile.update(
        dtype=rasterio.float32,
        count=image.shape[2],
        height=image.shape[0],
        width=image.shape[1]
    )

    with rasterio.open(output_path, 'w', **profile) as dst:
        for i in range(image.shape[2]):
            dst.write(image[:, :, i], i + 1)

    print(f"✅ Saved: {output_path}")

In [13]:
# =========================
# MAIN PIPELINE
# =========================
if __name__ == "__main__":
    vv_path = "/Users/parasningune/Downloads/14th_march-confirm_oil/2026-03-14-00:00_2026-03-14-23:59_Sentinel-1_IW_VV+VH_VV_(Raw).tiff"
    vh_path = "/Users/parasningune/Downloads/14th_march-confirm_oil/2026-03-14-00:00_2026-03-14-23:59_Sentinel-1_IW_VV+VH_VH_(Raw).tiff"

    # Load
    vv, vh, profile = load_sar(vv_path, vh_path)

    # Process
    image = process_sar(vv, vh)

    # Resize to 512x512 (like paper)
    #image_resized = resize_image(image, (512, 512))

    # Save
    save_tiff("processed_sar.tif", image, profile)

✅ Saved: processed_sar.tif
